# 그래프RAG를 5단계로 직접 만들기 (Hands-on)

**데이터:** 민원처리법 — **이미 조(條) 단위로 청킹된** 53청크 (`test2.json`)  
**모델:** `gpt-5-mini` (개체·관계 추출 / 커뮤니티 요약 / 답변 생성)

> 청킹은 이미 끝나 있다(구조 청킹, 조 단위). 그래서 이 실습은 **추출부터** 시작한다.

각 STEP은 앞 STEP의 출력을 입력으로 받는다:

```
test2.json → [S1] triples.json → [S2] graph.graphml
           → [S3] communities.json → [S4] summaries.json → [S5] 답변
```


## 0. 준비 — 설치 · 클라이언트 · 데이터 로드

- LLM 단계(STEP 1·4·5)만 API 키가 필요하다.
- gpt-5 계열은 **Responses API**(`client.responses.create`)로 호출한다.
- 청킹은 이미 완료됐으므로 여기선 **불러오기만** 한다.

In [1]:
%pip install -q openai networkx

Note: you may need to restart the kernel to use updated packages.


In [1]:
import os, re, json, getpass
from openai import OpenAI
import networkx as nx
from networkx.algorithms.community import greedy_modularity_communities

# API 키 (환경변수에 없으면 입력)
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OPENAI_API_KEY: ")

client = OpenAI()
MODEL = "gpt-5-mini"

def ask_llm(prompt: str) -> str:
    """gpt-5 계열은 Responses API 사용. 결과 텍스트만 돌려준다."""
    resp = client.responses.create(
        model=MODEL,
        input=prompt,
        # reasoning={"effort": "low"},  # 빠르고 싸게 돌리려면 주석 해제
    )
    return resp.output_text

def parse_json(text: str):
    """LLM 출력에서 JSON만 안전하게 뽑아 파싱."""
    text = text.strip().strip("`")
    text = re.sub(r"^json", "", text).strip()
    m = re.search(r"\[.*\]", text, re.S)     # 배열 우선
    return json.loads(m.group(0) if m else text)


### 데이터 로드 — 이미 나눠둔 조(條) 단위 청크

청킹은 이미 완료(구조 청킹). 메타데이터(`chapter`/`article_no`)가 장→조 계층까지 준다.

In [2]:
raw = json.load(open(r"C:\Users\User\Desktop\test2.json", encoding="utf-8"))

chunks = [{
    "id":      d["metadata"].get("article_no") or f"chunk_{i}",   # 제1조
    "title":   d["metadata"].get("article_title", ""),           # 목적
    "chapter": d["metadata"].get("chapter", ""),                 # 제1장 총칙
    "text":    d["pageContent"],
} for i, d in enumerate(raw)]

print(f"로드된 청크: {len(chunks)}개 (이미 청킹 완료)")
for c in chunks[:5]:
    print(f"[{c['id']}] {c['title']} · {c['chapter']}")

로드된 청크: 53개 (이미 청킹 완료)
[제1조] 목적 · 제1장 총칙
[제2조] 정의 · 제1장 총칙
[제3조] 적용 범위 · 제1장 총칙
[제4조] 민원 처리 담당자의 의무와 보호 · 제1장 총칙
[제5조] 민원인의 권리와 의무 · 제1장 총칙


## STEP 1 · 개체·관계 추출 — 조문 → 트리플

각 조문을 `gpt-5-mini`에 넣어 **(개체, 관계, 개체)** 트리플을 뽑는다.  
법령용으로 **정의 · 참조 · 주체행위** 세 종류를 노리고, 어느 조에서 나왔는지 **출처(source)** 를 붙인다.

> 53개 전체는 시간이 걸리니, 먼저 2개 조문으로 결과를 확인한 뒤 전체를 돌린다.

In [3]:
EXTRACT_PROMPT = """다음은 대한민국 법령의 한 조문이다.
여기서 (개체, 관계, 개체) 형태의 트리플을 추출하라.
- 정의: 조가 용어를 정의   예) ["제2조","정의","민원"]
- 참조: 다른 조문을 가리킴  예) ["제3조","적용예외","제36조제3항"]
- 주체행위: 주체의 권리·의무·행위  예) ["민원인","권리","신속한 응답"]
반드시 JSON 배열만 출력하라. 각 원소는 [개체, 관계, 개체] 3요소 배열.

[조문]
"""

def extract_triples(chunk):
    out = ask_llm(EXTRACT_PROMPT + chunk["text"])
    try:
        triples = parse_json(out)
    except Exception:
        return []
    # 3요소 트리플만 남기고 출처(조 번호)를 붙인다
    return [t + [chunk["id"]] for t in triples
            if isinstance(t, list) and len(t) == 3]

# 먼저 제2조·제3조로 테스트
for c in chunks[1:3]:
    print(f"--- {c['id']} {c['title']} ---")
    for t in extract_triples(c):
        print("  ", t)


--- 제2조 정의 ---
   ['제2조', '정의', '민원', '제2조']
   ['제2조', '정의', '민원인', '제2조']
   ['제2조', '정의', '행정기관', '제2조']
   ['제2조', '정의', '처분', '제2조']
   ['제2조', '정의', '복합민원', '제2조']
   ['제2조', '정의', '다수인관련민원', '제2조']
   ['제2조', '정의', '무인민원발급창구', '제2조']
   ['민원', '종류', '일반민원', '제2조']
   ['민원', '종류', '고충민원', '제2조']
   ['일반민원', '종류', '법정민원', '제2조']
   ['일반민원', '종류', '질의민원', '제2조']
   ['일반민원', '종류', '건의민원', '제2조']
   ['일반민원', '종류', '기타민원', '제2조']
   ['고충민원', '참조', '부패방지 및 국민권익위원회의 설치와 운영에 관한 법률 제2조제5호', '제2조']
   ['처분', '참조', '행정절차법 제2조제2호', '제2조']
   ['민원인', '제외', '행정기관(사경제의 주체로서 제기하는 경우는 제외)', '제2조']
   ['민원인', '제외', '행정기관과 사법상 계약관계에 있는 자(민원과 직접 관련된 계약관계만 해당)', '제2조']
   ['민원인', '제외', '성명·주소 등이 불명확한 자', '제2조']
   ['민원인', '제외', '대통령령으로 정하는 자', '제2조']
   ['공공기관', '참조', '공공기관의 운영에 관한 법률 제4조', '제2조']
   ['지방공사·지방공단', '참조', '지방공기업법', '제2조']
   ['학교', '참조', '초·중등교육법·고등교육법', '제2조']
   ['복합민원', '정의', '여러 관계 기관 또는 관계 부서의 인가·허가·승인·추천·협의 또는 확인 등을 거쳐 처리되는 법정민원', '제2조']
   ['다수인관련민원', '정의', '5세대 이상의 공동이해와 관련되어 5

In [4]:
# 전체 추출 → triples.json  (시간이 오래 걸리므로, chunks[:10] 등으로 줄여도 됨)
TARGET = chunks[:10]            # 예: chunks[:10]

all_triples = []
for c in TARGET:
    all_triples += extract_triples(c)
    print(f"{c['id']}: 누적 {len(all_triples)}개")

json.dump(all_triples, open("triples.json", "w", encoding="utf-8"),
          ensure_ascii=False, indent=2)
print("\n총 트리플:", len(all_triples), "→ triples.json 저장")


제1조: 누적 4개
제2조: 누적 34개
제3조: 누적 41개
제4조: 누적 47개
제5조: 누적 56개
제6조: 누적 62개
제7조: 누적 64개
제7조의2: 누적 69개
제8조: 누적 74개
제8조의2: 누적 79개

총 트리플: 79 → triples.json 저장


## STEP 2 · 그래프 조립 — 트리플 → networkx

트리플의 양끝(개체)은 **노드**, 관계는 **엣지**.  
같은 이름은 한 노드로 자동 병합되어, 여러 조를 관통하는 개체가 **허브**로 드러난다.

In [5]:
triples = json.load(open("triples.json", encoding="utf-8"))

G = nx.Graph()
for h, r, t, src in triples:
    G.add_edge(h, t, label=r, source=src)

print(f"노드 {G.number_of_nodes()}개 · 엣지 {G.number_of_edges()}개")
print("허브(연결 많은 순):")
for n, d in sorted(G.degree, key=lambda x: -x[1])[:5]:
    print(f"  {n} ({d})")

nx.write_graphml(G, "graph.graphml")


노드 97개 · 엣지 79개
허브(연결 많은 순):
  민원인 (16)
  제2조 (10)
  행정기관 (6)
  공공기관 (6)
  행정기관의 장 (6)


## STEP 3 · 커뮤니티 탐지 — 밀집 군집으로 나누기

서로 촘촘히 연결된 노드 무리를 자동으로 묶는다(모듈러리티 최적화).  
각 커뮤니티가 하나의 **주제**가 되고, 이것이 STEP 4 요약의 단위가 된다.

> 여기선 networkx 내장 `greedy_modularity_communities`를 쓴다.  
> 정식 Leiden을 쓰려면 `pip install leidenalg python-igraph`.

In [6]:
comms = list(greedy_modularity_communities(G))

communities = []
for i, c in enumerate(comms):
    communities.append({"id": f"C{i}", "nodes": list(c)})
    print(f"C{i} ({len(c)}개): " + ", ".join(list(c)[:6]) + (" …" if len(c) > 6 else ""))

json.dump(communities, open("communities.json", "w", encoding="utf-8"),
          ensure_ascii=False, indent=2)


C0 (16개): 성명·주소 등이 불명확한 자, 민원을 처리하는 담당자의 적법한 민원처리를 위한 요청에 협조할 의무, 공정한 응답을 받을 권리, 행정기관과 사법상 계약관계에 있는 자(민원과 직접 관련된 계약관계만 해당), 행정기관에 부당한 요구를 하는 행위, 행정기관에 민원을 신청할 권리 …
C1 (11개): 공공기관의 운영에 관한 법률 제4조에 따른 법인·단체·기관, 특별법에 따라 설립된 특수법인, 법령 또는 자치법규에 따라 행정권한이 있거나 행정권한을 위임·위탁받은 법인·단체·기관·개인, 초·중등교육법·고등교육법 및 그 밖의 다른 법률에 따라 설치된 각급 학교, 행정기관, 중앙행정기관 및 그 소속 기관 …
C2 (10개): 전화, 구술, 법정민원, 민원, 기타민원, 고충민원 …
C3 (8개): 복합민원, 관계법령등, 처분, 다수인관련민원, 무인민원발급창구, 제2조 …
C4 (7개): 행정기관의 장, 민원 처리 담당자의 신체적·정신적 피해의 예방 및 치료 등 대통령령으로 정하는 필요한 조치를 하여야 함, 민원의 내용 및 민원인·민원에 포함된 특정인의 개인정보 누설 방지 위한 필요한 조치 강구, 수집된 정보가 민원 처리의 목적 외의 용도로 사용되지 않도록 할 것, 법령의 규정 또는 위임이 없는 한 민원 처리의 절차 등을 강화하지 않음, 민원 처리를 지연시키지 않음 …
C5 (6개): 제42조, 제38조, 제39조제2항부터 제6항까지, 제37조, 제2조제3호가목의 국회ㆍ법원ㆍ헌법재판소ㆍ중앙선거관리위원회의 행정사무를 처리하는 기관, 제36조제3항
C6 (5개): 민원의 공정하고 적법한 처리, 이 법, 민원행정제도의 합리적 개선, 민원 처리에 관한 기본적인 사항, 국민의 권익 보호
C7 (4개): 민원 처리 담당자의 자긍심 제고, 매년 11월 24일, 민원에 대한 이해와 인식 제고, 민원의 날
C8 (3개): 제4조제3항, 제4조제4항, 제4조제2항
C9 (3개): 민원 처리를 지연시키는 사유가 아님, 관계법령등에서 정한 처리기간이 남아 있음, 그 민원과 관련 없는

In [9]:
!pip install -q pyvis

In [10]:
# 군집 시각화

from pyvis.network import Network

node_to_comm = {}

for i, c in enumerate(comms):
    for node in c:
        node_to_comm[node] = i

net = Network(
    height="750px",
    width="100%",
    bgcolor="#ffffff",
    font_color="#222222",
    notebook=False
)

# 노드 추가
for node in G.nodes():
    comm_id = node_to_comm.get(node, -1)
    
    net.add_node(
        node,
        label=node,
        title=f"노드: {node}<br>커뮤니티: C{comm_id}<br>연결 수: {G.degree(node)}",
        group=comm_id,
        size=15 + G.degree(node) * 3
    )

# 엣지 추가
for u, v, data in G.edges(data=True):
    label = data.get("label", "")
    
    net.add_edge(
        u,
        v,
        label=label,
        title=label
    )

net.show_buttons(filter_=["physics"])
net.write_html(r"C:\Users\User\Desktop\communities_graph.html")

print("communities_graph.html 파일이 생성되었습니다.")

communities_graph.html 파일이 생성되었습니다.


## STEP 4 · 커뮤니티 요약 — 각 군집을 한 단락으로

커뮤니티마다 그 안의 개체·관계를 `gpt-5-mini`에 주고 요약을 만든다.  
이 요약이 **Global 검색**의 재료 — 전역 질문이 오면 원문을 다 뒤지지 않고 요약만 종합한다.

In [7]:
SUMMARY_PROMPT = "다음 개체와 관계 목록을 한국어 한 단락으로 요약하라.\n\n"

summaries = {}
for c in communities:
    # 커뮤니티 내부 관계를 텍스트로
    rels = [f"{h} -[{d['label']}]- {t}"
            for h, t, d in G.edges(data=True)
            if h in c["nodes"] and t in c["nodes"]]
    text = "개체: " + ", ".join(c["nodes"]) + "\n관계: " + "; ".join(rels)
    summaries[c["id"]] = ask_llm(SUMMARY_PROMPT + text)
    print(f"[{c['id']}] {summaries[c['id']][:70]}...")

json.dump(summaries, open("summaries.json", "w", encoding="utf-8"),
          ensure_ascii=False, indent=2)


[C0] 민원인은 행정기관(다만 사경제의 주체로서 제기하는 경우는 제외), 민원과 직접 관련된 행정기관과의 사법상 계약관계 당사자, 성...
[C1] 개체·관계 목록을 요약하면, 행정기관은 국회·법원·헌법재판소·중앙선거관리위원회의 행정사무를 처리하는 기관, 중앙행정기관 및 그...
[C2] 민원은 이 법에서 정하는 바에 따라 적용되며, 종류로 일반민원과 고충민원으로 구분된다. 일반민원은 다시 법정민원·질의민원·건의...
[C3] 제2조는 처분, 복합민원, 다수인관련민원, 무인민원발급창구 및 관계법령등을 정의하고 있으며, 이 조문은 관련 규정으로서 행정절...
[C4] 행정기관의 장은 대통령령으로 정하는 바에 따라 민원 처리 담당자의 신체적·정신적 피해 예방 및 치료 등 필요한 조치를 하여야 ...
[C5] 제2조 제3호 가목에 따른 국회·법원·헌법재판소·중앙선거관리위원회의 행정사무를 처리하는 기관은 제36조 제3항, 제37조, 제...
[C6] 이 법은 민원 처리에 관한 기본적인 사항을 규정하며, 그 목적은 민원의 공정하고 적법한 처리와 민원행정제도의 합리적 개선 및 ...
[C7] 민원의 날은 매년 11월 24일로 지정되어 있으며, 민원에 대한 이해와 인식을 제고하고 민원 처리 담당자의 자긍심을 높이는 것...
[C8] 제4조의 제3항·제4항·제2항 세 개의 조문 가운데 제3항이 제2항과 제4항을 참조하고 있어, 제3항의 내용이 제2항 및 제4...
[C9] 관계법령등에서 정한 처리기간이 남아 있는 경우는 민원 처리를 지연시키는 사유에 해당하지 않으며, 이와 같은 ‘민원 처리를 지연...
[C10] 민원의 신청은 문서로 할 수 있는 신청방법이며, 해당 문서는 전자정부법 제2조 제7호에 규정된 전자문서를 포함할 수 있다....
[C11] 개체인 "전자문서"는 법령 관련 항목으로서 『전자정부법』 제2조 제7호를 참조하고 있으며, 주어진 두 관계는 동일한 법 조항을...
[C12] 다음과 같이 요약할 수 있다. 다른 법률에 민원에 관하여 별도의 특별 규정이 있는 

## STEP 5 · 질의 — Local vs Global

- **Local**: 관련 개체의 이웃 서브그래프만 모아 답 (사실형 질문)
- **Global**: 커뮤니티 요약들을 map→reduce로 종합 (전역·주제 질문)

벡터 top-k가 못 하는 **전역·조문 참조망** 질문이 여기서 산다.

In [11]:
def local_search(entity, question):
    if entity not in G:
        return f"'{entity}'는 그래프에 없습니다."
    nbrs = [f"{entity} -[{G[entity][n]['label']}]- {n}" for n in G.neighbors(entity)]
    ctx = "\n".join(nbrs)
    return ask_llm(f"다음 그래프 근거만으로 답하라.\n{ctx}\n\n질문: {question}")

def global_search(question):
    joined = "\n".join(f"[{k}] {v}" for k, v in summaries.items())
    return ask_llm(f"다음은 문서의 주제별 요약이다.\n{joined}\n\n"
                   f"이를 종합해 답하라.\n질문: {question}")

print("=== Local ===")
print(local_search("행정기관", "행정기관은 무엇과 어떻게 연결되어 있나?"))

print("\n=== Global ===")
print(global_search("민원처리법의 전체 구조를 종합해서 설명해줘."))


=== Local ===
그래프에 따르면:
- 행정기관은 제2조에서 정의(연결: 정의)된다.
- 행정기관은 다음을 포함(연결: 포함)한다:
  1. 국회·법원·헌법재판소·중앙선거관리위원회의 행정사무를 처리하는 기관
  2. 중앙행정기관 및 그 소속 기관
  3. 지방자치단체 및 그 소속 기관
  4. 공공기관
  5. 법령 또는 자치법규에 따라 행정권한이 있거나 행정권한을 위임·위탁받은 법인·단체·기관·개인

=== Global ===
아래는 주어진 요약들(C0–C20)을 바탕으로 민원처리법(이하 “법”)의 전체 구조를 정리한 내용입니다. 핵심 체계와 주요 조항들이 어떻게 연결되는지, 실무상 유의할 점을 포함해 요약합니다.

1. 목적과 적용범위
- 목적: 민원의 공정·적법한 처리, 민원행정제도의 합리적 개선, 국민 권익 보호(요약 C6).
- 적용범위: 행정기관이 처리하는 민원에 적용하되, 다른 법률에 특별 규정이 있는 경우 그 특별규정이 우선 적용되어 일반 규정의 적용이 배제될 수 있음(요약 C12, C15).

2. 주요 개념 정의(구조적 토대)
- 행정기관: 국회·법원·헌법재판소·중앙선거관리위원회의 행정사무를 처리하는 기관, 중앙행정기관·지방자치단체 및 소속 기관, 공공기관, 법령·자치법규에 따라 행정권한을 가진 또는 위임·위탁받은 법인·단체·기관·개인 등(요약 C1).
- 민원·관련 용어: 처분, 복합민원, 다수인관련민원, 무인민원발급창구 등 기본 정의를 규정하고 있으며(요약 C3), 관련 규정으로 행정절차법 등 타 법령과의 연계를 명시함.
- 전자문서·전자화문서: 전자정부법의 정의를 참조(요약 C10, C11, C19).

3. 민원인(신청인)의 권리와 의무
- 권리: 행정기관에 민원을 신청할 권리, 신속·공정·친절·적법한 응답을 받을 권리(요약 C0).
- 제외 대상: 행정기관 자체(사경제 주체로 제기하는 경우 제외), 민원과 직접 관련된 행정기관과의 사법상 계약관계 당사자, 신원 불명확자 등 대통령령으로 정하는 자는 민원인이 될 수 없음(요약 C

## 마무리 — 벡터 RAG와 비교해보기

같은 질문을 벡터 top-k RAG에도 던져보고 비교하면 그래프RAG의 가치가 드러난다.

- **국소 질문** ("제8조는 무엇인가?") → 벡터 RAG로도 충분
- **전역 질문** ("민원의 종류를 모두 종합하면?") → Global 검색이 우세
- **참조망 질문** ("제3조가 적용을 배제하는 조문들은?") → 그래프가 아니면 답하기 어렵다

> sLM 모델로 테스트를 해보고 싶다면, openrouter로 불러오기